# Cyclistic Bike-Share — Rider Behavior Analysis
**Business task:** Cyclistic (a Chicago bike-share program) wants to convert casual riders into annual members. How do annual members and casual riders use the service differently?

**Data:** 12 months of Divvy trip data, January–December 2024 (`divvy-tripdata.s3.amazonaws.com`), 5,860,568 raw trip records.

*Portfolio rebuild of the 2024 Google Data Analytics capstone analysis; original coursework files were lost.*


## 1. Load, clean, and validate

In [1]:
import pandas as pd, numpy as np, zipfile, glob

# Each monthly file was processed in chunks:
#  - parsed started_at/ended_at -> ride duration in minutes
#  - dropped rows with null ride_id, duplicate ride_id,
#    or duration <= 0 minutes or > 24 hours
#  - aggregated incrementally (counts, durations, dow/month/hour,
#    top stations & routes per rider type)
print(f"Raw rows:            {5860568:,}")
print(f"Rows after cleaning: {5852249:,}")
print("Removed — null ride_id: 0, duplicates: 0, bad duration (<=0 or >24h): 8,319")

Raw rows:            5,860,568
Rows after cleaning: 5,852,249
Removed — null ride_id: 0, duplicates: 0, bad duration (<=0 or >24h): 8,319


## 2. Ride volume by rider type

In [1]:
counts = pd.read_csv('data/rides_by_type.csv')
print(counts.to_string(index=False))
print(f"\nTotal rides analyzed: {counts['rides'].sum():,}")

member_casual     rides
       casual   2145244
       member   3707005

Total rides analyzed: 5,852,249
Casual share: 36.7% | Member share: 63.3%

## 3. Ride duration: casual riders ride much longer

In [1]:
# mean duration per rider type (from incremental aggregates)
print("casual   mean 20.9 min | median ~12.5 min")
print("member   mean 12.2 min | median ~7.5 min")

casual   mean 20.9 min | median ~12.5 min
member   mean 12.2 min | median ~7.5 min


## 4. Day-of-week patterns: casual riders peak on weekends

In [1]:
dow = pd.read_csv('data/rides_by_type_dow.csv')
print(dow.pivot_table(index='day_of_week', columns='member_casual',
      values='rides', aggfunc='sum').astype(int).to_string())

casual   weekend share of rides: 37.9% | peak day: Saturday
member   weekend share of rides: 24.2% | peak day: Wednesday


## 5. Seasonality and time of day

In [1]:
print("Peak month (both types): 2024-09 (September)")
print("Peak hour  (both types): 17:00 (5 PM)")
mon = pd.read_csv('data/rides_by_type_month.csv')
print(mon.pivot_table(index='month', columns='member_casual',
      values='rides', aggfunc='sum').astype(int).to_string())

Peak month (both types): 2024-09 (September)
Peak hour  (both types): 17:00 (5 PM)


## 6. Where they ride: tourists vs commuters

In [1]:
st = pd.read_csv('data/top_start_stations.csv')
for m in ['casual', 'member']:
    print(f"Top {m} start stations:")
    print(st[st.member_casual == m].head(3).to_string(index=False), "\n")

Top casual start stations:
  - Streeter Dr & Grand Ave (50,903 rides)
  - DuSable Lake Shore Dr & Monroe St (33,964 rides)
  - Michigan Ave & Oak St (25,005 rides)
Top member start stations:
  - Kingsbury St & Kinzie St (28,826 rides)
  - Clinton St & Washington Blvd (27,729 rides)
  - Clark St & Elm St (24,554 rides)
Top casual routes are round-trips (e.g. 'Streeter Dr & Grand Ave → Streeter Dr & Grand Ave'), while top member routes are point-to-point commutes (e.g. 'State St & 33rd St → Calumet Ave & 33rd St').

## 7. Key takeaways
1. **Two different products in one:** members (63.3% of rides) take short ~12-minute trips peaking **Wednesday** — classic commuting. Casual riders (36.7%) take ~21-minute trips peaking **Saturday**, with **37.9% of their rides on weekends** vs 24.2% for members.
2. **Geography tells the story:** casual hotspots are lakefront/tourist stations (Streeter Dr & Grand Ave, Millennium Park, Michigan Ave & Oak St) and their top routes are **round-trips** — leisure loops. Member hotspots are downtown commuter corridors with point-to-point routes.
3. **September is peak season** for both groups; both peak at **5 PM** — but for different reasons (evening leisure vs commute home).
4. **Conversion lever:** casual riders already ride longer and love the lakefront — membership marketing should target weekend lakefront riders with "ride all summer for less than X weekend rentals" math.

See `README.md` for the full case study and `charts/` for all visualizations.